# Inventory Operations Copilot - Capstone (Google Colab Edition)

This notebook runs the **complete capstone, Phase 1 to Phase 9**, inside Google Colab.

It exists because corporate networks running a TLS-inspecting proxy (for example Zscaler)
break the HTTPS call to `api.together.xyz` with
`CERTIFICATE_VERIFY_FAILED: unable to get local issuer certificate`.
Colab has direct internet access, so the live Together AI phases work without any
certificate workaround.

| Phase | What it demonstrates | Needs an API key? |
| --- | --- | --- |
| 1 | Problem framing, success criteria, fixed 10-scenario test set | No |
| 2 | Rule-based baseline agent + safe metadata logging | No |
| 3 | LangChain + Together AI prompt comparison (3 prompt variants) | Optional (`--offline` works) |
| 4 | Retrieval grounding over approved business documents | Optional (`--offline` works) |
| 5 | Read-only tool routing with refusal and tool-failure handling | No |
| 6 | Multi-turn planning, memory and context reset | No |
| 7 | Feedback capture and deterministic response adaptation | No |
| 8 | Deployment boundary: health, metrics, graceful failure, safe logs | No |
| 9 | Evaluation harness, accuracy/safety metrics and report | No |

**Run order:** execute the cells from top to bottom. Steps 1-4 are setup; the rest are the phases.

---
## Step 1 - Put the project into the Colab runtime

Pick **one** of the three options below and run only that cell.

### Option A - Upload a ZIP (simplest, no Google Drive needed)

On your Windows machine, right-click the `CapStoneProject_GoogleColab` folder and choose
**Send to > Compressed (zipped) folder**, then run the cell below and select that ZIP file.

In [ ]:
# Option A: upload CapStoneProject_GoogleColab.zip from your local machine
from google.colab import files
import zipfile, pathlib

uploaded = files.upload()                      # choose the .zip you created
for name in uploaded:
    with zipfile.ZipFile(name) as archive:
        archive.extractall('/content')
    print(f'Extracted {name} into /content')

for item in sorted(pathlib.Path('/content').iterdir()):
    print(' ', item)

### Option B - Mount Google Drive

Copy the `CapStoneProject_GoogleColab` folder into *My Drive* first, then run this cell.

In [ ]:
# Option B: mount Google Drive and point at the project folder
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/CapStoneProject_GoogleColab'   # edit if needed
import os
print('Found project:', os.path.isdir(PROJECT_PATH))

### Option C - Clone from GitHub

Use this if you pushed `CapStoneProject_GoogleColab` to a repository.

In [ ]:
# Option C: clone the repository
!git clone https://github.com/<your-user>/<your-repo>.git /content/CapStoneProject_GoogleColab

---
## Step 2 - Bootstrap the project

`colab_setup.py` finds the project root, adds it to `sys.path`, and sets the working directory.
It searches `/content`, `/content/drive/MyDrive` and the current folder automatically, so in most
cases you can call `bootstrap()` with no arguments.

In [ ]:
import sys, glob

# Locate colab_setup.py wherever the project landed, then import it.
candidates = (glob.glob('/content/**/colab_setup.py', recursive=True)
              + glob.glob('/content/drive/MyDrive/**/colab_setup.py', recursive=True))
assert candidates, 'colab_setup.py not found - re-run Step 1.'
project_root = candidates[0].rsplit('/', 1)[0]
sys.path.insert(0, project_root)

import colab_setup as cs
ROOT = cs.bootstrap(project_root)

---
## Step 3 - Install dependencies

Only Phase 3 and Phase 4 need third-party packages (`langchain-core`, `langchain-together`).
Every other phase is standard library only. Skip this cell if you will run everything offline.

In [ ]:
cs.install_requirements()

---
## Step 4 - Configure the Together AI credentials

**Recommended:** store the key in Colab Secrets instead of pasting it into a cell.

1. Click the **key icon** in the Colab left sidebar (*Secrets*).
2. Click **Add new secret**.
3. Name it exactly `TOGETHER_API_KEY`, paste your Together AI key as the value.
4. Enable **Notebook access** for this notebook.

If the secret is missing, the cell falls back to a hidden `getpass` prompt.
The key is never written to disk and never printed.

In [ ]:
HAS_KEY = cs.configure_together(
    model='meta-llama/Llama-3.3-70B-Instruct-Turbo',
    base_url='https://api.together.xyz/v1',
)
print('Live Together AI phases enabled:', HAS_KEY)

In [ ]:
# Optional connectivity check: confirms Colab can reach Together AI (no proxy interception).
import os, urllib.request

request = urllib.request.Request(
    'https://api.together.xyz/v1/models',
    headers={'Authorization': f"Bearer {os.environ.get('TOGETHER_API_KEY', '')}"},
)
try:
    with urllib.request.urlopen(request, timeout=30) as response:
        print('Together AI reachable. HTTP', response.status)
except Exception as exc:
    print('Connectivity/auth problem:', type(exc).__name__, exc)

---
# Phase 1 - Problem framing, success criteria and the fixed test set

Phase 1 is a design phase: it defines the business problem, the safety boundary
(decision support only, never act), measurable success criteria, and the 10 fixed
scenarios reused by every later phase.

In [ ]:
cs.show_file('Solution/phase1/problem-framing.md', max_lines=40)

In [ ]:
cs.show_file('Solution/phase1/success-criteria.md', max_lines=40)

In [ ]:
cs.show_file('Solution/phase1/test-scenarios.yaml', max_lines=None)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(cs.phase_dir(1) / 'workflow.png')))

---
# Phase 2 - Rule-based baseline agent

A deterministic, no-LLM baseline. It classifies the question, answers from approved
synthetic records, refuses write actions, and logs metadata only (no raw prompts).

In [ ]:
cs.run_phase(2, 'run_demo.py', save_as='run_demo.output')

In [ ]:
cs.run_phase_tests(2)

In [ ]:
cs.show_file('Solution/phase2/limitations.md', max_lines=None)

---
# Phase 3 - LangChain + Together AI prompt comparison

The same Phase 1 test set is run through three prompt variants (`basic`, `structured`,
`safety_focused`). `safety_focused` is the chosen default.

Run the offline cell first to prove the pipeline, then the live cell to call Together AI.

In [ ]:
# Offline: deterministic responses, no API key, no network.
cs.run_phase(3, 'run_comparison.py', ['--offline'],
             save_as='offline-prompt-comparison.out')

In [ ]:
# Live: calls Together AI through LangChain. Requires Step 3 and Step 4.
assert HAS_KEY, 'No TOGETHER_API_KEY configured - re-run Step 4.'
cs.run_phase(3, 'run_comparison.py', save_as='together_api_prompt.output')

In [ ]:
cs.show_file('Solution/phase3/outputs/prompt-comparison.md', max_lines=30)

In [ ]:
cs.run_phase_tests(3)

---
# Phase 4 - Retrieval grounding

An in-memory vector store over approved business documents. Each question is answered
twice - without retrieval and with retrieval - so the grounding improvement is visible,
including the refusal path when no document is relevant enough.

In [ ]:
cs.run_phase(4, 'run_retrieval_demo.py', ['--offline'],
             save_as='offline-retrieval-demo.out')

In [ ]:
# Live: Together AI composes the grounded answer from retrieved evidence.
assert HAS_KEY, 'No TOGETHER_API_KEY configured - re-run Step 4.'
cs.run_phase(4, 'run_retrieval_demo.py', save_as='together-retrieval-demo.out')

In [ ]:
cs.run_phase_tests(4)

---
# Phase 5 - Read-only tool routing

The copilot selects a read-only tool, refuses any write/action request, and handles a
failing tool call without guessing.

In [ ]:
cs.run_phase(5, 'run_demo.py', save_as='run_demo.output')

In [ ]:
cs.run_phase_tests(5)

---
# Phase 6 - Planning and memory

Multi-turn conversation: the follow-up question ("Why is it delayed?") resolves against
remembered context, and `reset` clears that context so the same question must escalate.

In [ ]:
cs.run_phase(6, 'run_demo.py', save_as='run_demo.output')

In [ ]:
cs.run_phase_tests(6)

---
# Phase 7 - Feedback and adaptation

A low rating with a reason is captured, and the next answer to the same question adapts
deterministically - no model retraining, no hidden state.

In [ ]:
cs.run_phase(7, 'run_demo.py', save_as='run_demo.output')

In [ ]:
cs.run_phase_tests(7)

---
# Phase 8 - Deployment boundary

A framework-free service façade: health check, request handling, latency/error metrics,
graceful failure, and PII-redacted structured logs.

In [ ]:
cs.run_phase(8, 'run_local.py', save_as='run_local.output')

In [ ]:
cs.show_file('Solution/phase8/local-events.jsonl', max_lines=10)

In [ ]:
cs.run_phase_tests(8)

---
# Phase 9 - Evaluation harness

Replays the Phase 1 scenarios and produces accuracy, safety and escalation metrics
plus a human-readable report.

In [ ]:
cs.run_phase(9, 'run_evaluation.py', save_as='run_evaluation.output')

In [ ]:
cs.show_file('Solution/phase9/reports/evaluation-report.md', max_lines=None)

In [ ]:
cs.run_phase_tests(9)

---
# Run everything in one shot

`run_all_phases.py` executes Phase 1 to Phase 9 plus every unit test suite and writes
each console log into the matching `Solution/phase*/outputs/` folder.

Use `--offline` for a network-free run, or `--live` to exercise Together AI in Phase 3 and 4.

In [ ]:
!python run_all_phases.py --offline

In [ ]:
!python run_all_phases.py --live

---
# Collect and download the generated artifacts

Bundles every `outputs/`, `logs/` and `reports/` folder into a single ZIP.

In [ ]:
archive = cs.zip_outputs()

In [ ]:
from google.colab import files
files.download(str(archive))

---
## Troubleshooting

| Symptom | Cause | Fix |
| --- | --- | --- |
| `colab_setup.py not found` | Step 1 did not place the project under `/content` | Re-run Step 1, then re-run Step 2 |
| `Could not find the capstone root` | Project root passed explicitly is wrong | `cs.bootstrap('/content/CapStoneProject_GoogleColab')` |
| `TOGETHER_API_KEY is not set` | Secret missing or notebook access disabled | Re-run Step 4; check the Secrets panel toggle |
| `ModuleNotFoundError: langchain_together` | Step 3 skipped | Run `cs.install_requirements()` |
| `CERTIFICATE_VERIFY_FAILED` | You are running on the corporate machine, not Colab | Run this notebook in Colab, or use `--offline` locally |
| Live cell returns `401` | Invalid or expired Together AI key | Regenerate the key at api.together.ai and update the secret |